In [42]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
import joblib

In [28]:
# Load updated datasets
fights_df = pd.read_csv('Fights.csv')
fighters_df = pd.read_csv('Fighters.csv')

# Prepare fighter suffixes
fighters_1 = fighters_df.add_suffix('_1')
fighters_2 = fighters_df.add_suffix('_2')

# Merge fighter 1 statistics
df = pd.merge(
    fights_df, 
    fighters_1, 
    left_on='Fighter_Id_1', 
    right_on='Fighter_Id_1', 
    how='left'
)

# Merge fighter 2 statistics
df = pd.merge(
    df, 
    fighters_2, 
    left_on='Fighter_Id_2', 
    right_on='Fighter_Id_2', 
    how='left'
)

df.head()

,Fight_Id,Fighter_Id_1,Fighter_Id_2,Fighter_1,Fighter_2,KD_1,KD_2,STR_1,STR_2,TD_1,...,Full Name_2,Nickname_2,Ht._2,Wt._2,Reach_2,Stance_2,W_2,L_2,D_2,Belt_2
0,4a0db214d9721d6e,d661ce4da776fc20,c03520b5c88ed6b4,Merab Dvalishvili,Petr Yan,0,0,134,139,2,...,Merab Dvalishvili,The Machine,5.6,135.0,68.0,Orthodox,21,4,0,True
1,dfa692db6d39330c,17e97649403ba428,a0f0004aadf10b71,Alexandre Pantoja,Joshua Van,0,0,6,2,0,...,Alexandre Pantoja,The Cannibal,5.5,125.0,67.0,Orthodox,30,5,0,True
2,fbbb9e72900b71f5,4461d7e47375a895,792be9a24df82ed6,Brandon Moreno,Tatsuro Taira,0,0,9,28,0,...,Brandon Moreno,The Assassin Baby,5.7,125.0,70.0,Orthodox,23,8,2,False
3,1dc29f4c6fcdd356,6e743a33d56bdaa4,056c493bbd76a918,Henry Cejudo,Payton Talbott,0,1,60,134,1,...,Henry Cejudo,Triple C,5.4,135.0,64.0,Orthodox,16,5,0,False
4,6d6ab10cbaa45e8c,99df7d0a2a08a8a8,ef5dcb10d2bd4b0f,Jan Blachowicz,Bogdan Guskov,1,1,84,80,0,...,Bogdan Guskov,Czarevitch,6.3,205.0,76.0,Orthodox,18,3,0,False


In [29]:
print(df.shape)

print(df.isnull().sum())

(8461, 67)
Fight_Id        0
Fighter_Id_1    0
Fighter_Id_2    0
Fighter_1       0
Fighter_2       0
               ..
Stance_2        0
W_2             0
L_2             0
D_2             0
Belt_2          0
Length: 67, dtype: int64


In [30]:
# 1. Filter out draws ('D') to keep binary outcomes only
df_clean = df[df['Result_1'].isin(['W', 'L'])].copy()

# 2. Create the binary target variable (1 for Fighter 1 Win, 0 for Fighter 1 Loss)
df_clean['Winner'] = (df_clean['Result_1'] == 'W').astype(int)

# 3. Select pre-fight features (Attributes known BEFORE the fight starts)
feature_cols = [
    'Ht._1', 'Wt._1', 'Reach_1', 'W_1', 'L_1',
    'Ht._2', 'Wt._2', 'Reach_2', 'W_2', 'L_2',
    'Weight_Class', 'Stance_1', 'Stance_2'
]

# 4. Create the final modeling DataFrame with features and target
df_model = df_clean[feature_cols + ['Winner']].copy()

# 5. Display the shape and first few rows of the cleaned dataset
print("Filtered Dataset Shape:", df_model.shape)
df_model.head()

Filtered Dataset Shape: (8311, 14)


,Ht._1,Wt._1,Reach_1,W_1,L_1,Ht._2,Wt._2,Reach_2,W_2,L_2,Weight_Class,Stance_1,Stance_2,Winner
0,5.7,135.0,67.0,20.0,5.0,5.6,135.0,68.0,21,4,Bantamweight,Switch,Orthodox,0
1,5.5,125.0,65.0,16.0,2.0,5.5,125.0,67.0,30,5,Flyweight,Orthodox,Orthodox,0
2,5.7,125.0,70.0,18.0,1.0,5.7,125.0,70.0,23,8,Flyweight,Orthodox,Orthodox,0
3,5.1,135.0,70.0,11.0,1.0,5.4,135.0,64.0,16,5,Bantamweight,Switch,Orthodox,0
5,5.1,155.0,73.0,17.0,3.0,5.1,155.0,72.0,23,2,Lightweight,Switch,Switch,0


In [31]:
print(df_model.isnull().sum())

Ht._1             6
Wt._1             3
Reach_1         305
W_1               1
L_1               1
Ht._2            14
Wt._2             8
Reach_2         956
W_2               0
L_2               0
Weight_Class      0
Stance_1          1
Stance_2          0
Winner            0
dtype: int64


In [32]:
# 1. List of numerical columns to impute using grouped median
num_cols = ['Ht._1', 'Wt._1', 'Reach_1', 'Ht._2', 'Wt._2', 'Reach_2', 'W_1', 'L_1']

# Impute missing values in numerical columns using median per Weight_Class
for col in num_cols:
    df_model[col] = df_model.groupby('Weight_Class')[col].transform(lambda x: x.fillna(x.median()))

# 2. Impute missing value in categorical column (Stance_1) using Mode
stance_1_mode = df_model['Stance_1'].mode()[0]
df_model['Stance_1'] = df_model['Stance_1'].fillna(stance_1_mode)

# 3. Verify that all missing values have been resolved
print("Missing values count after imputation:")
print(df_model.isnull().sum())

df_model.head()

Missing values count after imputation:
Ht._1           0
Wt._1           0
Reach_1         0
W_1             0
L_1             0
Ht._2           0
Wt._2           0
Reach_2         1
W_2             0
L_2             0
Weight_Class    0
Stance_1        0
Stance_2        0
Winner          0
dtype: int64


,Ht._1,Wt._1,Reach_1,W_1,L_1,Ht._2,Wt._2,Reach_2,W_2,L_2,Weight_Class,Stance_1,Stance_2,Winner
0,5.7,135.0,67.0,20.0,5.0,5.6,135.0,68.0,21,4,Bantamweight,Switch,Orthodox,0
1,5.5,125.0,65.0,16.0,2.0,5.5,125.0,67.0,30,5,Flyweight,Orthodox,Orthodox,0
2,5.7,125.0,70.0,18.0,1.0,5.7,125.0,70.0,23,8,Flyweight,Orthodox,Orthodox,0
3,5.1,135.0,70.0,11.0,1.0,5.4,135.0,64.0,16,5,Bantamweight,Switch,Orthodox,0
5,5.1,155.0,73.0,17.0,3.0,5.1,155.0,72.0,23,2,Lightweight,Switch,Switch,0


In [33]:
# Fill any remaining single null value with the overall median of Reach_2
df_model['Reach_2'] = df_model['Reach_2'].fillna(df_model['Reach_2'].median())

# Verify everything is strictly 0
print(df_model.isnull().sum())

Ht._1           0
Wt._1           0
Reach_1         0
W_1             0
L_1             0
Ht._2           0
Wt._2           0
Reach_2         0
W_2             0
L_2             0
Weight_Class    0
Stance_1        0
Stance_2        0
Winner          0
dtype: int64


In [34]:
# ==========================================
# FEATURE ENGINEERING (RATES & DIFFERENTIALS)
# ==========================================

# 1. Ensure df_clean is assigned to df_model
df_model = df_clean.copy()

# 2. Calculate Win Rates to prevent raw wins bias
df_model['Win_Rate_1'] = df_model['W_1'] / (df_model['W_1'] + df_model['L_1'] + 1e-5)
df_model['Win_Rate_2'] = df_model['W_2'] / (df_model['W_2'] + df_model['L_2'] + 1e-5)

# 3. Calculate Differential Features (Fighter 1 minus Fighter 2)
df_model['Diff_Win_Rate'] = df_model['Win_Rate_1'] - df_model['Win_Rate_2']
df_model['Diff_Reach'] = df_model['Reach_1'] - df_model['Reach_2']
df_model['Diff_Ht'] = df_model['Ht._1'] - df_model['Ht._2']
df_model['Diff_Total_Fights'] = (df_model['W_1'] + df_model['L_1']) - (df_model['W_2'] + df_model['L_2'])

# 4. Drop redundant raw counters to focus the model on relative ratios
df_model.drop(columns=['W_1', 'L_1', 'W_2', 'L_2', 'Win_Rate_1', 'Win_Rate_2'], inplace=True)

print("Engineered features added successfully!")
df_model.head()

Engineered features added successfully!


,Fight_Id,Fighter_Id_1,Fighter_Id_2,Fighter_1,Fighter_2,KD_1,KD_2,STR_1,STR_2,TD_1,...,Wt._2,Reach_2,Stance_2,D_2,Belt_2,Winner,Diff_Win_Rate,Diff_Reach,Diff_Ht,Diff_Total_Fights
0,4a0db214d9721d6e,d661ce4da776fc20,c03520b5c88ed6b4,Merab Dvalishvili,Petr Yan,0,0,134,139,2,...,135.0,68.0,Orthodox,0,True,0,-0.040000,-1.0,0.1,0.0
1,dfa692db6d39330c,17e97649403ba428,a0f0004aadf10b71,Alexandre Pantoja,Joshua Van,0,0,6,2,0,...,125.0,67.0,Orthodox,0,True,0,0.031746,-2.0,0.0,-17.0
2,fbbb9e72900b71f5,4461d7e47375a895,792be9a24df82ed6,Brandon Moreno,Tatsuro Taira,0,0,9,28,0,...,125.0,70.0,Orthodox,2,False,0,0.205433,0.0,0.0,-12.0
3,1dc29f4c6fcdd356,6e743a33d56bdaa4,056c493bbd76a918,Henry Cejudo,Payton Talbott,0,1,60,134,1,...,135.0,64.0,Orthodox,0,False,0,0.154762,6.0,-0.3,-9.0
5,c797b58d85c840a4,2e7878927067fdca,99bd51917728c25d,Grant Dawson,Manuel Torres,0,1,8,9,0,...,155.0,72.0,Switch,1,False,0,-0.070000,1.0,0.0,-5.0


In [37]:
# Select categorical columns to encode
categorical_cols = ['Weight_Class', 'Stance_1', 'Stance_2']

# Apply One-Hot Encoding to prepare all features for model input
df_encoded = pd.get_dummies(df_model, columns=categorical_cols, drop_first=True, dtype=int)

# Display the shape and preview of the final dataset
print("Encoded Dataset Shape:", df_encoded.shape)
df_encoded.head()

Encoded Dataset Shape: (8311, 87)


,Fight_Id,Fighter_Id_1,Fighter_Id_2,Fighter_1,Fighter_2,KD_1,KD_2,STR_1,STR_2,TD_1,...,Weight_Class_Women's Flyweight,Weight_Class_Women's Strawweight,Stance_1_Orthodox,Stance_1_Sideways,Stance_1_Southpaw,Stance_1_Switch,Stance_2_Orthodox,Stance_2_Sideways,Stance_2_Southpaw,Stance_2_Switch
0,4a0db214d9721d6e,d661ce4da776fc20,c03520b5c88ed6b4,Merab Dvalishvili,Petr Yan,0,0,134,139,2,...,0,0,0,0,0,1,1,0,0,0
1,dfa692db6d39330c,17e97649403ba428,a0f0004aadf10b71,Alexandre Pantoja,Joshua Van,0,0,6,2,0,...,0,0,1,0,0,0,1,0,0,0
2,fbbb9e72900b71f5,4461d7e47375a895,792be9a24df82ed6,Brandon Moreno,Tatsuro Taira,0,0,9,28,0,...,0,0,1,0,0,0,1,0,0,0
3,1dc29f4c6fcdd356,6e743a33d56bdaa4,056c493bbd76a918,Henry Cejudo,Payton Talbott,0,1,60,134,1,...,0,0,0,0,0,1,1,0,0,0
5,c797b58d85c840a4,2e7878927067fdca,99bd51917728c25d,Grant Dawson,Manuel Torres,0,1,8,9,0,...,0,0,0,0,0,1,0,0,0,1


In [38]:
# 1. Separate Features (X) and Target (y)
X = df_encoded.drop(columns=['Winner'])
y = df_encoded['Winner']

# Select ONLY numeric columns to avoid String conversion errors in StandardScaler
X_numeric = X.select_dtypes(include=['int64', 'float64', 'int32', 'float32']).copy()

# 2. Split the dataset into Training (80%) and Testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(
    X_numeric, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Initialize StandardScaler
scaler = StandardScaler()

# 4. Fit on Training data and transform both Train and Test sets
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Display shapes to verify the split
print("X_train shape:", X_train_scaled.shape)
print("X_test shape:", X_test_scaled.shape)

X_train shape: (6648, 67)
X_test shape: (1663, 67)


In [43]:
from sklearn.naive_bayes import GaussianNB

# 1. Cleaning remaining NaN values using SimpleImputer
imputer = SimpleImputer(strategy='median')
X_train_clean = imputer.fit_transform(X_train_scaled)
X_test_clean = imputer.transform(X_test_scaled)

# 2. Define the 4 core classification models
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(),
    'Naive Bayes': GaussianNB()
}

# 3. Train and evaluate models
results = []

for name, model in models.items():
    model.fit(X_train_clean, y_train)
    y_pred = model.predict(X_test_clean)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    
    results.append({
        'Model': name,
        'Accuracy': round(acc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1-Score': round(f1, 4)
    })

# Display comparison results
results_df = pd.DataFrame(results)
results_df.sort_values(by='Accuracy', ascending=False, inplace=True)
results_df.reset_index(drop=True, inplace=True)
results_df

,Model,Accuracy,Precision,Recall,F1-Score
0,Random Forest,0.8737,0.8805,0.9299,0.9045
1,Decision Tree,0.8274,0.8690,0.8617,0.8653
2,K-Nearest Neighbors,0.8196,0.8513,0.8720,0.8615
3,Naive Bayes,0.3921,0.9683,0.0570,0.1077


In [44]:
# 1. Train and save the winning Random Forest model
best_model = RandomForestClassifier(n_estimators=100, random_state=42)
best_model.fit(X_train_clean, y_train)

# 2. Save the trained model, scaler, and imputer
joblib.dump(best_model, 'ufc_rf_model.pkl')
joblib.dump(scaler, 'ufc_scaler.pkl')
joblib.dump(imputer, 'ufc_imputer.pkl')

print("Best model (Random Forest), StandardScaler, and Imputer saved successfully!")

Best model (Random Forest), StandardScaler, and Imputer saved successfully!
